# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoder-only:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
- Decoder-based:
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)

In [3]:
import numpy as np

import torch
from datasets import load_dataset
import pandas as pd

import datasets

## Baseline models

In [4]:
from transformers import AutoTokenizer, AutoModel

In [5]:
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")
model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [6]:
 tokenizer.num_special_tokens_to_add(pair=False)

2

In [7]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size, overlap):
    real_chunks_size = chunk_size - tokenizer.num_special_tokens_to_add(pair=False)    # for each chunk special tokens will be appened after

    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks

    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)

    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        max_length=chunk_size,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, number_of_chunks):
    # outputs shape: [ num_chunks * num_texts, chunk_size, *]
    # converting to [num_texts, num_chunks, chunk_size, *]
    re_grouped = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks
        re_grouped.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        text_starts_i = text_ends_i
    return re_grouped

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="longest",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized

In [8]:
inputs = [
    "I like to leave work after my eight-hour tea-break.",
    "The murder hornet was disappointed by the preconceived ideas people had of him.",
    "The underground bunker was filled with chips and candy."
]

In [9]:
tokenized_outer_batch, number_of_chunks = tokenize_chunking_strategy(tokenizer, inputs, 5, 0)

In [10]:
tokenized_outer_batch["input_ids"]

tensor([[     0,     87,   1884,     47,      2],
        [     0,  31358,   4488,   7103,      2],
        [     0,    759, 136659,      9,      2],
        [     0,  56816,  26156,      2,      1],
        [     0,     20,  70751,      5,      2],
        [     0,    581, 162882,   3328,      2],
        [     0,   2043,    509, 242980,      2],
        [     0,    390,     70, 118562,      2],
        [     0,      6,   3956,     14,      2],
        [     0,  25647,   3395,   1902,      2],
        [     0,    111,   4049,      5,      2],
        [     0,    581,   1379,  64330,      2],
        [     0, 191645,    509, 152382,      2],
        [     0,    678, 110537,    136,      2],
        [     0,   7841,     53,      5,      2]])

In [11]:
class XMLRoBERTa():

    name = "xlm-roberta-large"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large", dtype=torch.float32)
        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

        self.model.eval()

    def encode(
            self,
            inputs,
            strategy,
            **kwargs) -> torch.tensor:

        if strategy == "chunking":
            tokenized, numbers_of_tokens = tokenize_chunking_strategy(self.tokenizer, inputs, self.chunk_size, 0)
        if strategy == "first":
            tokenized = tokenize_first_startegy(self.tokenizer, inputs, self.chunk_size)

        with torch.inference_mode():
            outputs = self.model(**tokenized)

            if strategy == "chunking":
                re_grouped = re_group_chunked_outputs(outputs.last_hidden_state, number_of_chunks)
                embeddings = [t.mean(dim=1).mean(dim=0) for t in re_grouped]
                return embeddings
            if strategy == "first":
                embeddings = outputs.last_hidden_state.mean(dim=1)
                return embeddings


class Qwen3_Embedding():

    name = "Qwen3-Embedding-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size

        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float32)

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states):
        return last_hidden_states[:, -1]

    def encode(
            self,
            text,
            strategy = "chunking",
            **kwargs) -> torch.tensor:

        if strategy == "chunking":
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, inputs, self.chunk_size, 0)
        if strategy == "first":
            tokenized = tokenize_first_startegy(self.tokenizer, inputs, self.chunk_size)
            print(tokenized.size())

        with torch.inference_mode():
            outputs = self.model(**tokenized)

            if strategy == "chunking":
                re_grouped = re_group_chunked_outputs(outputs.last_hidden_state, numbers_of_chunks)
                embeddings = [self.__get_eos_token_embedding(t).mean(dim=0) for t in re_grouped]
                return embeddings
            if strategy == "first":
                embeddings = self.__get_eos_token_embedding(outputs.last_hidden_state)
                return embeddings

In [24]:
inputs

['I like to leave work after my eight-hour tea-break.',
 'The murder hornet was disappointed by the preconceived ideas people had of him.',
 'The underground bunker was filled with chips and candy.']

In [26]:
xlm_roberta.encode(inputs, "first")

tensor([[ 0.0699, -0.1156,  0.1139,  ..., -0.0027, -0.0538,  0.0149],
        [ 0.0311, -0.0818,  0.0475,  ..., -0.0206,  0.0361, -0.0450],
        [ 0.0971, -0.0602,  0.1422,  ...,  0.0364, -0.0594, -0.0435]])

## LongEmbed LEMBWikimQARetrieval

In [12]:
from datasets import load_dataset

ds = load_dataset("dwzhu/LongEmbed", name="2wikimqa")
corpus = ds["corpus"]
queries = ds["queries"]
qrels = ds["qrels"]

README.md: 0.00B [00:00, ?B/s]

2wikimqa/corpus.jsonl:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

queries.jsonl: 0.00B [00:00, ?B/s]

2wikimqa/qrels.jsonl:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

Generating corpus split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating queries split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating qrels split:   0%|          | 0/300 [00:00<?, ? examples/s]

In [13]:
qwen3_embed = Qwen3_Embedding(512)
xlm_roberta = XMLRoBERTa(512)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

### Encoding document with each model with each text-preprocess strategy

In [22]:
# encode each document in a corpus
def encode_documents(corpus, model, strategy, batch_size=10):
    print(f"Encoding a coprus of length {len(corpus)}, processing using batches with size {batch_size}.")
    document_embeddings = {}
    start = 0
    for batch_end in range(batch_size, len(corpus), batch_size):
        batch = corpus[start:batch_end]
        print(f"Processing the batch [{start+1}:{batch_end}]")
        start = batch_end

        embedding = model.encode(batch["text"], strategy)

        for i, doc_id in enumerate(batch["doc_id"]):
            document_embeddings[doc_id] = embedding[i].numpy()

    return document_embeddings

# encode each query
def encode_queries(queries, model, batch_size=10):
    queries_embeddings = {}
    print("Encoding a set of queries.")
    start = 0
    for batch_end in range(batch_size, len(queries), batch_size):
        batch = queries[start:batch_end]
        print(f"Processing the batch [{start+1}:{batch_end}]")
        start = batch_end

        print(len(batch["text"]))
        embedding = model.encode(batch["text"], "first")

        print(embedding.size())

        for i, q_id in enumerate(batch["qid"]):
            queries_embeddings[q_id] = embedding[i].numpy()

    return queries_embeddings

In [23]:
# encoding the queries with each mode
# strategy does not really matter in this case since the query will be one chunk long anyway
q3_query_embed = encode_queries(queries, qwen3_embed)
roberta_query_embed = encode_queries(queries, xlm_roberta)

Encoding a set of queries.
Processing the batch [1:10]
10
torch.Size([3, 1024])


IndexError: index 3 is out of bounds for dimension 0 with size 3

In [ ]:
pd.DataFrame(q3_query_embed).to_csv("./query_embed/q3_query_embed.csv")
pd.DataFrame(roberta_query_embed).to_csv("./query_embed/roberta_query_embed.csv")

In [ ]:
q3_doc_embed_chunking = encode_documents(corpus, qwen3_embed, strategy="chunking")
roberta_doc_embed_chunking = encode_documents(corpus, xlm_roberta, strategy="chunking")

Token indices sequence length is longer than the specified maximum sequence length for this model (4234 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
pd.DataFrame(q3_doc_embed_chunking).to_csv("q3_doc_embed_chunking.csv")
pd.DataFrame(roberta_doc_embed_chunking).to_csv("roberta_doc_embed_chunking.csv")

In [ ]:
q3_doc_embed_first = encode_documents(corpus, qwen3_embed, strategy="first")
roberta_doc_embed_first = encode_documents(corpus, xlm_roberta, strategy="first")

In [ ]:
pd.DataFrame(q3_doc_embed_first).to_csv("q3_doc_embed_first.csv")
pd.DataFrame(roberta_doc_embed_first).to_csv("roberta_doc_embed_first.csv")

In [ ]:
q3_doc_embed_last = encode_documents(corpus, qwen3_embed, strategy="last")
roberta_doc_embed_last = encode_documents(corpus, xlm_roberta, strategy="last")

In [ ]:
pd.DataFrame(q3_doc_embed_last).to_csv("q3_doc_embed_last.csv")
pd.DataFrame(roberta_doc_embed_last).to_csv("roberta_doc_embed_last.csv")

### Calculating evaluation metrics

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
def MAP_at_K(documents_embed, queries_embed, qrel, k):
    sum_ap_at_k = 0
    n_queries = 0
    for q in qrel:
        q_id = q["qid"]
        true_doc_id = q["doc_id"]

        q_emebedding = np.array(queries_embed[q_id])
        similarities = []
        for doc_id in documents_embed:
            doc_embedding = np.array(documents_embed[doc_id])
            cos_sim = cosine_similarity(q_emebedding.reshape(1, -1), doc_embedding.reshape(1, -1))[0][0]
            similarities.append({"doc_id": doc_id, "similarity": cos_sim})

        similarities_df = (
            pd.DataFrame(similarities)
            .sort_values(by="similarity", ascending=False)
        )
        top_k = similarities_df["doc_id"].to_list()[:k]

        # assuming that there is only one relevant document for the query
        try:
            true_document_position = top_k.index(true_doc_id) + 1
            ap_at_k = 1 / true_document_position
        except ValueError:
            ap_at_k = 0
        sum_ap_at_k += ap_at_k
        n_queries += 1
    map = sum_ap_at_k / n_queries
    return map

In [ ]:
def mean_nDCG_at_k(documents_embed, queries_embed, qrel, k):
    sum_ndcg_at_k = 0
    n_queries = 0

    idcg_at_k = 1
    for i in range(1, 10+1):
        idcg_at_k += 1 / np.log2(i + 2)

    for q in qrel:
        q_id = q["qid"]
        true_doc_id = q["doc_id"]

        q_emebedding = np.array(queries_embed[q_id])
        similarities = []
        for doc_id in documents_embed:
            doc_embedding = np.array(documents_embed[doc_id])
            cos_sim = cosine_similarity(q_emebedding.reshape(1, -1), doc_embedding.reshape(1, -1))[0][0]
            similarities.append({"doc_id": doc_id, "similarity": cos_sim})

        similarities_df = (
            pd.DataFrame(similarities)
            .sort_values(by="similarity", ascending=False)
        )
        top_k = similarities_df["doc_id"].to_list()[:k]

        # assuming that there is only one relevant document for the query
        try:
            true_document_position = top_k.index(true_doc_id) + 1
            dcg_at_k = 1 / np.log2(true_document_position + 1)
        except ValueError:
            dcg_at_k = 0
        ndcg_at_k = dcg_at_k / idcg_at_k
        sum_ndcg_at_k += ndcg_at_k
        n_queries += 1

    mean_ndcg = sum_ndcg_at_k / n_queries
    return mean_ndcg

In [ ]:
roberta_doc_embeddings = pd.read_csv("./doc_embeddings/roberta_doc_embed_chunking.csv").drop(columns="Unnamed: 0").to_dict(orient="list")
q3_doc_embeddings = pd.read_csv("./doc_embeddings/q3_doc_embed_chunking.csv").drop(columns="Unnamed: 0").to_dict(orient="list")

roberta_query_embeddings = pd.read_csv("./query_embeddings/roberta_query_embed.csv").drop(columns="Unnamed: 0").to_dict(orient="list")
q3_query_embeddings = pd.read_csv("./query_embeddings/q3_query_embed.csv").drop(columns="Unnamed: 0").to_dict(orient="list")

In [ ]:
print("XLM-RoBERTa-large nDCG@k: ", mean_nDCG_at_k(roberta_doc_embeddings, roberta_query_embeddings, qrels, 10))
print("XLM-RoBERTa-large MAP@k: ", MAP_at_K(roberta_doc_embeddings, roberta_query_embeddings, qrels, 10))

XLM-RoBERTa-large nDCG@k:  0.027452690239719697
XLM-RoBERTa-large MAP@k:  0.10360449735449732


In [ ]:
print("Qwen3 Embedding nDCG@k: ", mean_nDCG_at_k(q3_doc_embeddings, q3_query_embeddings, qrels, 10))
print("Qwen3 Embedding MAP@k: ", MAP_at_K(q3_doc_embeddings, q3_query_embeddings, qrels, 10))

Qwen3 Embedding nDCG@k:  0.15436043151119166
Qwen3 Embedding MAP@k:  0.7148743386243387
